# ECG Classification - Best CNN Model (v2)

## Model Overview
This notebook implements a **Convolutional Neural Network (CNN)** for ECG classification.

**Architecture**: 4 convolutional blocks (32→64→128→256 filters)
- **Input**: 188 columns of ECG time series data (representing one heartbeat)
- **Output**: Binary classification (0 = normal, 1 = abnormal)

**Key Features of this Model**:
- Deeper network with batch normalization for stable training
- Learning rate scheduling (automatically reduces learning rate when model stops improving)
- Early stopping with restore best weights
- Class weight balancing to handle imbalanced data
- 4 convolutional blocks with increasing filter sizes (32→64→128→256)

---

## What is a CNN?
**For non-technical readers**: A CNN is like a pattern detector. Imagine looking at an ECG signal - a CNN automatically learns to recognize important patterns (like the shape of heartbeats, rhythm irregularities) that distinguish normal from abnormal heart activity. It does this by scanning small windows across the signal and learning which patterns matter most.

In [ ]:
# =============================================================================
# STEP 1: Import Libraries
# =============================================================================
# What this does: Loads all the tools we need for data processing, 
# model building, and evaluation

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score, roc_curve, precision_score, recall_score, f1_score,
    accuracy_score, log_loss, mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, davies_bouldin_score
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout,
    BatchNormalization, Activation
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')
print(f'TensorFlow version: {tf.__version__}')

---

## Step 2: Data Loading

**What this does**: Loads the ECG dataset from CSV files. Each row represents one heartbeat with 188 time points, plus a label (0=normal, 1=abnormal).

**For non-technical readers**: Think of this like loading patient records into a system. Each record contains the electrical activity of one heartbeat, measured at 188 different moments in time.

In [ ]:
# =============================================================================
# STEP 2: Data Loading
# =============================================================================
# Load data - supports both Kaggle and local environments

try:
    # Kaggle environment
    df1 = pd.read_csv('/kaggle/input/ecg-dataset/ecg.csv', header=None)
    df2 = pd.read_csv('/kaggle/input/ecg2-dataset/ecg3.csv', header=None)
    df2 = df2.rename(columns={0: 'orig_0'})
    df2.insert(0, 0, df2['orig_0'])
    df2.columns = range(df2.shape[1])
    df = pd.concat([df1, df2], ignore_index=True)
except:
    try:
        # Local environment - try repository root
        df1 = pd.read_csv('../../ecg.csv', header=None)
        df2 = pd.read_csv('../../ecg3.csv', header=None)
        df2 = df2.rename(columns={0: 'orig_0'})
        df2.insert(0, 0, df2['orig_0'])
        df2.columns = range(df2.shape[1])
        df = pd.concat([df1, df2], ignore_index=True)
    except:
        df = pd.read_csv('../../dataset_aritmia_NEW.csv')

print(f'Dataset shape: {df.shape}')
print(f'Total samples: {df.shape[0]}')
print(f'Features per sample: {df.shape[1] - 1}')

In [ ]:
# Add meaningful column names
n_features = df.shape[1] - 1
column_names = [f'f{i}' for i in range(n_features)] + ['label']
df.columns = column_names

print(f'Number of features (time points): {n_features}')
print('\nFirst 5 samples:')
df.head()

---

## Step 3: Exploratory Data Analysis

**What this does**: Examines the distribution of labels to understand class balance.

**For non-technical readers**: We're checking how many normal vs abnormal heartbeats we have. If one group is much larger than the other, we need to account for this during training so the model doesn't become biased.

In [ ]:
# =============================================================================
# STEP 3: Exploratory Data Analysis
# =============================================================================
# Check label distribution - important for understanding class imbalance

print('Label distribution:')
print(df['label'].value_counts())
print('\nPercentage:')
print(df['label'].value_counts(normalize=True) * 100)

# Visualize the distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Distribution of ECG Labels')
axes[0].set_xlabel('Label (0=Normal, 1=Abnormal)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal (0)', 'Abnormal (1)'], rotation=0)

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                  colors=['green', 'red'], labels=['Normal', 'Abnormal'])
axes[1].set_title('Percentage of ECG Labels')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

---

## Step 4: Data Preprocessing - NORMALIZATION

**What this does**: 
1. **Normalization (StandardScaler)**: Transforms all features to have mean=0 and standard deviation=1
2. **Reshaping**: Converts data into the format required by the CNN (samples × timesteps × features)
3. **Label Encoding**: Converts labels to one-hot encoded format for classification

**For non-technical readers**: 
- **Normalization** is like converting different currencies to a standard unit. ECG values might be in different ranges (e.g., 800-1200), and we convert them to a standard scale (-3 to +3 typically). This helps the model learn faster and more effectively.
- Without normalization, features with larger values might dominate the learning process unfairly.

In [ ]:
# =============================================================================
# STEP 4: Data Preprocessing - NORMALIZATION (CRITICAL STEP)
# =============================================================================
# Separate features and labels
X = df.drop('label', axis=1).values
y = df['label'].values

# =============================================================================
# NORMALIZATION using StandardScaler
# This is crucial for neural networks to ensure all features are on the same scale
# StandardScaler: transforms data to have mean=0 and std=1
# =============================================================================
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)

print('Normalization Statistics:')
print(f'Before normalization - Mean: {X.mean():.2f}, Std: {X.std():.2f}')
print(f'After normalization - Mean: {X_normalized.mean():.6f}, Std: {X_normalized.std():.6f}')

# Reshape for Conv1D: (samples, timesteps, features)
X_normalized = X_normalized.reshape((X_normalized.shape[0], X_normalized.shape[1], 1))

# Encode labels to categorical format
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

print(f'\nX shape after preprocessing: {X_normalized.shape}')
print(f'y shape after preprocessing: {y_categorical.shape}')
print(f'Classes: {le.classes_}')

---

## Step 5: Train-Validation-Test Split (80%-10%-10%)

**What this does**: Splits the data into three sets:
- **Training set (80%)**: Used to train the model
- **Validation set (10%)**: Used to monitor training and prevent overfitting
- **Test set (10%)**: Used for final evaluation (never seen during training)

**For non-technical readers**: 
- Think of it like preparing for an exam: Training data is your study material, validation data is practice tests to check progress, and test data is the actual exam.
- We use **stratification** to ensure each split has the same proportion of normal/abnormal samples.
- We use **random_state=42** to ensure reproducibility (same split every time we run the code).

In [ ]:
# =============================================================================
# STEP 5: Train-Validation-Test Split (80%-10%-10%)
# =============================================================================
# IMPORTANT: Using consistent random_state=42 for reproducibility across all notebooks
# This ensures fair model comparison

RANDOM_STATE = 42  # Fixed random state for reproducibility

# First split: 80% training, 20% temporary (for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_normalized, y_categorical, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y_encoded
)

# Get encoded labels for temp set (for stratification)
y_temp_encoded = np.argmax(y_temp, axis=1)

# Second split: 50% of temp = 10% validation, 50% of temp = 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp_encoded
)

print('Dataset Split Summary:')
print('=' * 50)
print(f'Total samples: {len(X_normalized)}')
print(f'Training samples: {len(X_train)} ({len(X_train)/len(X_normalized)*100:.1f}%)')
print(f'Validation samples: {len(X_val)} ({len(X_val)/len(X_normalized)*100:.1f}%)')
print(f'Test samples: {len(X_test)} ({len(X_test)/len(X_normalized)*100:.1f}%)')

# Verify stratification
print('\nLabel distribution in each set:')
print(f'Training - Class 0: {np.sum(np.argmax(y_train, axis=1)==0)}, Class 1: {np.sum(np.argmax(y_train, axis=1)==1)}')
print(f'Validation - Class 0: {np.sum(np.argmax(y_val, axis=1)==0)}, Class 1: {np.sum(np.argmax(y_val, axis=1)==1)}')
print(f'Test - Class 0: {np.sum(np.argmax(y_test, axis=1)==0)}, Class 1: {np.sum(np.argmax(y_test, axis=1)==1)}')

In [ ]:
# Compute class weights for handling imbalanced data
y_train_classes = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_classes), y=y_train_classes)
class_weight_dict = dict(enumerate(class_weights))

print('Class Weights (to handle imbalanced data):')
print(f'Class 0 (Normal): {class_weight_dict[0]:.4f}')
print(f'Class 1 (Abnormal): {class_weight_dict[1]:.4f}')
print('\nNote: Higher weight means the model pays more attention to that class')

---

## Step 5b: Data Leakage Verification

**What this does**: Performs comprehensive checks to ensure no data leakage between train/validation/test sets.

**Why this matters for non-technical readers**:
- **Data leakage** is when information from the test set accidentally "leaks" into training, causing artificially perfect results (like 100% accuracy)
- This is like a student getting the exam answers before the test - the results look great but aren't real
- These checks verify that our model's performance is genuine and will generalize to new data

**Checks performed**:
1. **No Overlap Check**: Ensures no sample appears in multiple sets
2. **Duplicate Check**: Finds duplicate samples within each set
3. **Label Distribution**: Verifies class balance is consistent
4. **Feature Statistics**: Confirms normalization was done correctly
5. **Shape Verification**: Validates data dimensions
6. **Random Sampling**: Confirms stratified sampling worked

In [ ]:
# =============================================================================
# STEP 5b: DATA LEAKAGE VERIFICATION & INTEGRITY CHECKS
# =============================================================================
# These checks ensure no data leakage between train/val/test sets
# and verify data integrity to prevent artificially high accuracy (1.00)

print('=' * 70)
print('DATA LEAKAGE & INTEGRITY VERIFICATION')
print('=' * 70)

# -----------------------------------------------------------------------------
# CHECK 1: Verify no sample overlap between train/val/test sets
# -----------------------------------------------------------------------------
print('\n[CHECK 1] Verifying NO OVERLAP between train/val/test sets...')

def check_array_overlap(arr1, arr2, name1, name2):
    """Check if any samples in arr1 exist in arr2"""
    arr1_flat = arr1.reshape(arr1.shape[0], -1)
    arr2_flat = arr2.reshape(arr2.shape[0], -1)
    
    # Convert to tuple of tuples for set comparison
    set1 = set(map(tuple, arr1_flat))
    set2 = set(map(tuple, arr2_flat))
    
    overlap = set1.intersection(set2)
    overlap_count = len(overlap)
    
    if overlap_count > 0:
        print(f'   ⚠️  WARNING: {overlap_count} overlapping samples between {name1} and {name2}!')
        return False
    else:
        print(f'   ✓ No overlap between {name1} and {name2}')
        return True

# Check all pairs
train_val_ok = check_array_overlap(X_train, X_val, 'Training', 'Validation')
train_test_ok = check_array_overlap(X_train, X_test, 'Training', 'Test')
val_test_ok = check_array_overlap(X_val, X_test, 'Validation', 'Test')

if train_val_ok and train_test_ok and val_test_ok:
    print('   ✓ PASSED: No data leakage detected between splits!')
else:
    print('   ⚠️  FAILED: Data leakage detected! Check your splitting logic.')

# -----------------------------------------------------------------------------
# CHECK 2: Verify no duplicate samples within each set
# -----------------------------------------------------------------------------
print('\n[CHECK 2] Checking for DUPLICATE samples within each set...')

def count_duplicates(arr, name):
    """Count duplicate samples in array"""
    arr_flat = arr.reshape(arr.shape[0], -1)
    unique_samples = set(map(tuple, arr_flat))
    total = len(arr_flat)
    unique = len(unique_samples)
    duplicates = total - unique
    
    if duplicates > 0:
        print(f'   ⚠️  WARNING: {name} has {duplicates} duplicate samples ({duplicates/total*100:.2f}%)')
        return duplicates
    else:
        print(f'   ✓ {name}: No duplicates ({total} unique samples)')
        return 0

train_dups = count_duplicates(X_train, 'Training set')
val_dups = count_duplicates(X_val, 'Validation set')
test_dups = count_duplicates(X_test, 'Test set')

# -----------------------------------------------------------------------------
# CHECK 3: Verify label distribution consistency (no severe imbalance leakage)
# -----------------------------------------------------------------------------
print('\n[CHECK 3] Verifying LABEL DISTRIBUTION consistency...')

train_labels = np.argmax(y_train, axis=1)
val_labels = np.argmax(y_val, axis=1)
test_labels = np.argmax(y_test, axis=1)

train_ratio = np.mean(train_labels)
val_ratio = np.mean(val_labels)
test_ratio = np.mean(test_labels)

print(f'   Training set - Class 1 ratio: {train_ratio:.4f}')
print(f'   Validation set - Class 1 ratio: {val_ratio:.4f}')
print(f'   Test set - Class 1 ratio: {test_ratio:.4f}')

# Check if ratios are similar (within 5% tolerance)
ratio_tolerance = 0.05
if abs(train_ratio - val_ratio) > ratio_tolerance or abs(train_ratio - test_ratio) > ratio_tolerance:
    print(f'   ⚠️  WARNING: Label distributions differ by more than {ratio_tolerance*100}%!')
else:
    print('   ✓ Label distributions are consistent across splits')

# -----------------------------------------------------------------------------
# CHECK 4: Feature statistics comparison (detect normalization leakage)
# -----------------------------------------------------------------------------
print('\n[CHECK 4] Comparing FEATURE STATISTICS across splits...')

train_mean = np.mean(X_train)
train_std = np.std(X_train)
val_mean = np.mean(X_val)
val_std = np.std(X_val)
test_mean = np.mean(X_test)
test_std = np.std(X_test)

print(f'   Training - Mean: {train_mean:.6f}, Std: {train_std:.6f}')
print(f'   Validation - Mean: {val_mean:.6f}, Std: {val_std:.6f}')
print(f'   Test - Mean: {test_mean:.6f}, Std: {test_std:.6f}')

# Check if stats are reasonably similar
if abs(train_mean - val_mean) > 0.5 or abs(train_mean - test_mean) > 0.5:
    print('   ⚠️  WARNING: Feature means differ significantly across splits!')
else:
    print('   ✓ Feature statistics are consistent (normalized correctly)')

# -----------------------------------------------------------------------------
# CHECK 5: Verify data shapes are correct
# -----------------------------------------------------------------------------
print('\n[CHECK 5] Verifying DATA SHAPES...')
print(f'   X_train shape: {X_train.shape}')
print(f'   X_val shape: {X_val.shape}')
print(f'   X_test shape: {X_test.shape}')
print(f'   y_train shape: {y_train.shape}')
print(f'   y_val shape: {y_val.shape}')
print(f'   y_test shape: {y_test.shape}')

# Verify proportions
total = len(X_train) + len(X_val) + len(X_test)
train_pct = len(X_train) / total * 100
val_pct = len(X_val) / total * 100
test_pct = len(X_test) / total * 100

print(f'\n   Split proportions: Train={train_pct:.1f}%, Val={val_pct:.1f}%, Test={test_pct:.1f}%')
if abs(train_pct - 80) > 2 or abs(val_pct - 10) > 2 or abs(test_pct - 10) > 2:
    print('   ⚠️  WARNING: Split proportions deviate from expected 80/10/10!')
else:
    print('   ✓ Split proportions match expected 80/10/10')

# -----------------------------------------------------------------------------
# CHECK 6: Sample a few indices to verify randomization
# -----------------------------------------------------------------------------
print('\n[CHECK 6] Verifying STRATIFIED SAMPLING...')
# Check that samples are from different parts of the original dataset

for name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    class_0 = np.sum(labels == 0)
    class_1 = np.sum(labels == 1)
    print(f'   {name}: Class 0 = {class_0}, Class 1 = {class_1}, Ratio = {class_1/(class_0+class_1):.4f}')

print('\n' + '=' * 70)
print('VERIFICATION COMPLETE')
print('=' * 70)

---

## Step 6: CNN Model Architecture

**What this does**: Builds the CNN model with 4 convolutional blocks.

**Architecture Explanation for non-technical readers**:
- **Convolutional Layers**: Like pattern detectors that scan across the ECG signal
- **Batch Normalization**: Helps the model train faster and more stably
- **MaxPooling**: Reduces the data size while keeping important patterns
- **Dropout**: Randomly turns off some neurons during training to prevent overfitting
- **Dense Layers**: Final layers that combine all patterns to make a prediction

In [ ]:
# =============================================================================
# STEP 6: CNN Model Architecture
# =============================================================================

def create_best_cnn_model(input_shape, num_classes):
    """
    Creates the Best CNN Model for ECG Classification.
    
    Architecture:
    - 4 Convolutional blocks with increasing filters (32→64→128→256)
    - Each block: Conv1D → BatchNorm → ReLU → Conv1D → BatchNorm → ReLU → MaxPool → Dropout
    - Global Average Pooling to reduce parameters
    - Dense layers for final classification
    
    This architecture is designed to capture both local patterns (individual wave shapes)
    and global patterns (overall rhythm) in ECG signals.
    """
    model = Sequential([
        # Block 1: Initial feature extraction with 32 filters
        Conv1D(32, kernel_size=5, padding='same', input_shape=input_shape),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(32, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        
        # Block 2: Deeper patterns with 64 filters
        Conv1D(64, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(64, kernel_size=5, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        
        # Block 3: Complex patterns with 128 filters
        Conv1D(128, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        Conv1D(128, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        
        # Block 4: Highest-level features with 256 filters
        Conv1D(256, kernel_size=3, padding='same'),
        BatchNormalization(),
        Activation('relu'),
        GlobalAveragePooling1D(),
        
        # Dense layers for classification
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    return model

# Create the model
model = create_best_cnn_model(
    input_shape=(X_train.shape[1], 1),
    num_classes=y_categorical.shape[1]
)

# Compile with optimizer and loss function
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
print('CNN Model Architecture:')
print('=' * 60)
model.summary()

---

## Step 7: Model Training

**What this does**: Trains the model on the training data while monitoring performance on validation data.

**Callbacks explained for non-technical readers**:
- **EarlyStopping**: Stops training if the model stops improving (like stopping practice when you've mastered the material)
- **ReduceLROnPlateau**: Slows down learning when stuck (like taking smaller steps when approaching a destination)
- **ModelCheckpoint**: Saves the best version of the model

In [ ]:
# =============================================================================
# STEP 7: Model Training
# =============================================================================

# Define callbacks for optimal training
callbacks = [
    # Save the best model based on validation loss
    ModelCheckpoint(
        'ecg_cnn_best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    # Stop training if no improvement for 15 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when learning plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
print('Starting Training...')
print('=' * 60)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print('\nTraining Complete!')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy Over Training', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss Over Training', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Step 7b: Post-Training Verification

**What this does**: Checks training results for signs of overfitting or data leakage.

**Warning Signs**:
- **100% accuracy**: Almost always indicates data leakage
- **Large train-val gap**: Suggests overfitting
- **Very low loss**: May indicate memorization rather than learning

In [ ]:
# =============================================================================
# STEP 7b: POST-TRAINING VERIFICATION
# =============================================================================
# Additional checks to detect potential issues after training

print('=' * 70)
print('POST-TRAINING VERIFICATION')
print('=' * 70)

# Check for suspiciously perfect accuracy
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
best_val_acc = max(history.history['val_accuracy'])

print(f'\n[CHECK 1] Final Training Accuracy: {final_train_acc:.4f}')
print(f'[CHECK 2] Final Validation Accuracy: {final_val_acc:.4f}')
print(f'[CHECK 3] Best Validation Accuracy: {best_val_acc:.4f}')

# Warning for suspiciously high accuracy
if final_train_acc >= 0.99:
    print('\n⚠️  WARNING: Training accuracy >= 99% - possible overfitting or data leakage!')
if final_val_acc >= 0.99:
    print('⚠️  WARNING: Validation accuracy >= 99% - verify no data leakage!')
if final_train_acc >= 1.0 or final_val_acc >= 1.0:
    print('⚠️  CRITICAL: Perfect 100% accuracy detected - likely data leakage!')

# Check for overfitting (large gap between train and val)
accuracy_gap = final_train_acc - final_val_acc
print(f'\n[CHECK 4] Train-Val Accuracy Gap: {accuracy_gap:.4f}')
if accuracy_gap > 0.1:
    print('⚠️  WARNING: Large accuracy gap suggests overfitting!')
elif accuracy_gap < -0.05:
    print('⚠️  WARNING: Validation > Training suggests data issues!')
else:
    print('✓ Accuracy gap is within normal range')

# Check loss convergence
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
print(f'\n[CHECK 5] Final Training Loss: {final_train_loss:.4f}')
print(f'[CHECK 6] Final Validation Loss: {final_val_loss:.4f}')

if final_train_loss < 0.01:
    print('⚠️  WARNING: Very low training loss - possible overfitting!')
if final_val_loss < 0.01:
    print('⚠️  WARNING: Very low validation loss - verify data integrity!')

print('\n' + '=' * 70)
print('POST-TRAINING VERIFICATION COMPLETE')
print('=' * 70)

---

## Step 8: Model Evaluation - Comprehensive Metrics

### Understanding the Metrics:

**Classification Metrics:**
1. **Accuracy**: Overall correct predictions / total predictions
2. **Precision**: Of all predicted positives, how many are actually positive?
3. **Recall (Sensitivity/TPR)**: Of all actual positives, how many did we catch?
4. **F1 Score**: Harmonic mean of precision and recall (balanced metric)
5. **Log Loss**: Measures confidence of predictions (lower is better)
6. **AUC-ROC**: Area under the ROC curve (1.0 is perfect, 0.5 is random)

**ROC Curve Components:**
- **TPR (True Positive Rate)**: Same as Recall
- **FPR (False Positive Rate)**: False alarms rate
- **TNR (True Negative Rate)**: Correctly identified negatives
- **FNR (False Negative Rate)**: Missed positives

**Error Metrics (treated as regression on probabilities):**
- **MAE, MSE, RMSE, RMSLE**: Measure prediction errors
- **R²**: Explained variance (usually for regression, but can be informative)

**Clustering Quality Metrics:**
- **Silhouette Score**: Measures how similar samples are to their own cluster
- **Davies-Bouldin Index**: Measures cluster separation (lower is better)

In [ ]:
# =============================================================================
# STEP 8: Comprehensive Model Evaluation on TEST SET
# =============================================================================

# Generate predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred_proba, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print('=' * 70)
print('COMPREHENSIVE MODEL EVALUATION - CNN (v2)')
print('=' * 70)

In [ ]:
# =============================================================================
# CATEGORY 1: Basic Classification Metrics
# =============================================================================
# What these measure: How well the model classifies samples into correct categories

print('\n' + '=' * 70)
print('CATEGORY 1: BASIC CLASSIFICATION METRICS')
print('=' * 70)
print('These metrics measure how well the model classifies ECG signals.')
print('-' * 70)

# 1. Accuracy
accuracy = accuracy_score(y_true_classes, y_pred_classes)
print(f'\n1. ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)')
print('   → Measures: Percentage of correct predictions out of all predictions')
print('   → Interpretation: Higher is better. 1.0 means perfect accuracy.')

# 2. Precision
precision = precision_score(y_true_classes, y_pred_classes, average='weighted')
print(f'\n2. PRECISION: {precision:.4f}')
print('   → Measures: Of all predicted positives, how many are actually positive?')
print('   → Interpretation: High precision = few false alarms')

# 3. Recall (Sensitivity)
recall = recall_score(y_true_classes, y_pred_classes, average='weighted')
print(f'\n3. RECALL (Sensitivity): {recall:.4f}')
print('   → Measures: Of all actual positives, how many did we correctly identify?')
print('   → Interpretation: High recall = we catch most abnormal cases')

# 4. F1 Score
f1 = f1_score(y_true_classes, y_pred_classes, average='weighted')
print(f'\n4. F1 SCORE: {f1:.4f}')
print('   → Measures: Harmonic mean of precision and recall')
print('   → Interpretation: Balanced metric - good when both precision & recall matter')

# 5. Log Loss (Logarithmic Loss)
logloss = log_loss(y_true_classes, y_pred_proba)
print(f'\n5. LOGARITHMIC LOSS (Log Loss): {logloss:.4f}')
print('   → Measures: How confident the model is in its predictions')
print('   → Interpretation: Lower is better. Penalizes confident wrong predictions.')

In [ ]:
# =============================================================================
# CATEGORY 2: ROC Curve and AUC
# =============================================================================
# What these measure: Model's ability to distinguish between classes at various thresholds

print('\n' + '=' * 70)
print('CATEGORY 2: ROC CURVE AND AUC METRICS')
print('=' * 70)
print('These metrics measure the model\'s ability to distinguish between classes.')
print('-' * 70)

# 6. AUC-ROC Score
auc_score = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print(f'\n6. AREA UNDER CURVE (AUC): {auc_score:.4f}')
print('   → Measures: Overall ability to distinguish between classes')
print('   → Interpretation: 1.0 = perfect, 0.5 = random guessing, >0.9 = excellent')

# Calculate ROC curve for the positive class (abnormal = 1)
fpr, tpr, thresholds = roc_curve(y_true_classes, y_pred_proba[:, 1])

# Calculate additional rates
tnr = 1 - fpr  # True Negative Rate (Specificity)
fnr = 1 - tpr  # False Negative Rate

# Find optimal threshold (Youden's J statistic)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f'\n   ROC Curve Components at Optimal Threshold ({optimal_threshold:.4f}):')
print(f'   • TPR (True Positive Rate/Sensitivity): {tpr[optimal_idx]:.4f}')
print(f'   • TNR (True Negative Rate/Specificity): {tnr[optimal_idx]:.4f}')
print(f'   • FPR (False Positive Rate): {fpr[optimal_idx]:.4f}')
print(f'   • FNR (False Negative Rate): {fnr[optimal_idx]:.4f}')

In [ ]:
# =============================================================================
# 6.5 & 6.6: ROC Curve Visualization
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 6.5 ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], c='green', s=100, 
                label=f'Optimal Point (threshold={optimal_threshold:.2f})', zorder=5)
axes[0].set_xlabel('False Positive Rate (FPR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (TPR)', fontsize=12)
axes[0].set_title('ROC Curve - CNN Model (v2)', fontsize=14)
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# Add annotation
axes[0].annotate('Perfect Classifier\n(0, 1)', xy=(0, 1), xytext=(0.2, 0.8),
                 fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

# 6.6 ROC-AUC Bar Chart (comparing with baselines)
models = ['Random\nClassifier', 'CNN Model\n(This Model)']
auc_values = [0.5, auc_score]
colors = ['red', 'green']

axes[1].bar(models, auc_values, color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(y=0.9, color='orange', linestyle='--', label='Excellent threshold (0.9)')
axes[1].set_ylabel('AUC Score', fontsize=12)
axes[1].set_title('AUC-ROC Comparison', fontsize=14)
axes[1].set_ylim([0, 1.1])
axes[1].legend()
for i, v in enumerate(auc_values):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('\n   ROC Curve Interpretation:')
print('   • The curve shows the trade-off between sensitivity and specificity')
print('   • A perfect classifier hugs the top-left corner')
print('   • The diagonal line represents random guessing')
# Provide interpretation based on AUC value
if auc_score >= 0.9:
    interpretation = 'excellent'
elif auc_score >= 0.8:
    interpretation = 'good'
elif auc_score >= 0.7:
    interpretation = 'fair'
else:
    interpretation = 'needs improvement'
print(f'   • Our model has AUC = {auc_score:.4f}, indicating {interpretation} performance')

In [ ]:
# =============================================================================
# CATEGORY 3: Confusion Matrix
# =============================================================================
# What it shows: Detailed breakdown of correct and incorrect predictions

print('\n' + '=' * 70)
print('CATEGORY 3: CONFUSION MATRIX')
print('=' * 70)
print('Shows the detailed breakdown of predictions vs actual values.')
print('-' * 70)

# 7. Confusion Matrix
cm = confusion_matrix(y_true_classes, y_pred_classes)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard confusion matrix
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal (0)', 'Abnormal (1)'])
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14)

# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_normalized, display_labels=['Normal (0)', 'Abnormal (1)'])
disp2.plot(ax=axes[1], cmap='Blues', values_format='.2%')
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14)

plt.tight_layout()
plt.show()

# Calculate and print confusion matrix metrics
tn, fp, fn, tp = cm.ravel()
print(f'\n   Confusion Matrix Breakdown:')
print(f'   • True Negatives (TN): {tn} - Correctly identified normal ECGs')
print(f'   • True Positives (TP): {tp} - Correctly identified abnormal ECGs')
print(f'   • False Positives (FP): {fp} - Normal ECGs incorrectly flagged as abnormal')
print(f'   • False Negatives (FN): {fn} - Abnormal ECGs missed (dangerous!)')
print(f'\n   Clinical Interpretation:')
print(f'   • Sensitivity (catching abnormalities): {tp/(tp+fn)*100:.1f}%')
print(f'   • Specificity (correctly clearing normal): {tn/(tn+fp)*100:.1f}%')

In [ ]:
# =============================================================================
# CATEGORY 4: Error Metrics (Regression-style metrics on probabilities)
# =============================================================================
# Note: These are typically for regression, but can be informative for classification
# when applied to prediction probabilities

print('\n' + '=' * 70)
print('CATEGORY 4: ERROR METRICS (MAE, MSE, RMSE, RMSLE, R²)')
print('=' * 70)
print('These metrics measure prediction errors (applied to class probabilities).')
print('Note: While typically used for regression, they provide insight into prediction quality.')
print('-' * 70)

# Use the probability of the true class as the target
y_true_proba = y_test[np.arange(len(y_true_classes)), y_true_classes]
y_pred_proba_true_class = y_pred_proba[np.arange(len(y_true_classes)), y_true_classes]

# 8a. Mean Absolute Error (MAE)
mae = mean_absolute_error(y_true_proba, y_pred_proba_true_class)
print(f'\n8a. MEAN ABSOLUTE ERROR (MAE): {mae:.6f}')
print('    → Measures: Average absolute difference between predicted and actual probabilities')
print('    → Interpretation: Lower is better. 0 means perfect probability predictions.')

# 8b. Mean Squared Error (MSE)
mse = mean_squared_error(y_true_proba, y_pred_proba_true_class)
print(f'\n8b. MEAN SQUARED ERROR (MSE): {mse:.6f}')
print('    → Measures: Average squared difference (penalizes large errors more)')
print('    → Interpretation: Lower is better.')

# 8c. Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)
print(f'\n8c. ROOT MEAN SQUARED ERROR (RMSE): {rmse:.6f}')
print('    → Measures: Square root of MSE (same units as the data)')
print('    → Interpretation: Lower is better.')

# 8d. Root Mean Squared Logarithmic Error (RMSLE)
# Add small constant to avoid log(0)
y_true_safe = np.clip(y_true_proba, 1e-10, 1)
y_pred_safe = np.clip(y_pred_proba_true_class, 1e-10, 1)
rmsle = np.sqrt(np.mean((np.log1p(y_pred_safe) - np.log1p(y_true_safe))**2))
print(f'\n8d. ROOT MEAN SQUARED LOGARITHMIC ERROR (RMSLE): {rmsle:.6f}')
print('    → Measures: RMSE on log scale (penalizes underestimates more)')
print('    → Interpretation: Lower is better.')

# 8e. R² Score (Coefficient of Determination)
r2 = r2_score(y_true_proba, y_pred_proba_true_class)
print(f'\n8e. R² SCORE (Coefficient of Determination): {r2:.6f}')
print('    → Measures: Proportion of variance explained by the model')
print('    → Interpretation: 1.0 is perfect, 0 means no better than mean prediction')

In [ ]:
# =============================================================================
# CATEGORY 5: Clustering Quality Metrics
# =============================================================================
# These metrics evaluate how well the model's predictions cluster the data

print('\n' + '=' * 70)
print('CATEGORY 5: CLUSTERING QUALITY METRICS')
print('=' * 70)
print('These metrics evaluate how well the predictions group similar samples together.')
print('-' * 70)

# Get the feature representation from the model (before the final classification layer)
# We'll use the predicted class labels for clustering evaluation

# Reshape X_test for clustering metrics
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# 9a. Silhouette Score
try:
    silhouette = silhouette_score(X_test_flat, y_pred_classes)
    print(f'\n9a. SILHOUETTE SCORE: {silhouette:.4f}')
    print('    → Measures: How similar samples are to their own cluster vs other clusters')
    print('    → Interpretation: Range [-1, 1]. Higher is better. >0.5 is good clustering.')
except Exception as e:
    print(f'\n9a. SILHOUETTE SCORE: Could not compute - {e}')

# 9b. Davies-Bouldin Index
try:
    dbi = davies_bouldin_score(X_test_flat, y_pred_classes)
    print(f'\n9b. DAVIES-BOULDIN INDEX: {dbi:.4f}')
    print('    → Measures: Average similarity between clusters (lower is better)')
    print('    → Interpretation: 0 is perfect separation. Lower values indicate better clustering.')
except Exception as e:
    print(f'\n9b. DAVIES-BOULDIN INDEX: Could not compute - {e}')

In [ ]:
# =============================================================================
# SUMMARY: All Metrics at a Glance
# =============================================================================

print('\n' + '=' * 70)
print('SUMMARY: ALL EVALUATION METRICS - CNN MODEL (v2)')
print('=' * 70)

print('\n┌─────────────────────────────────────────────────────────────────────┐')
print('│ CLASSIFICATION METRICS                                              │')
print('├─────────────────────────────────────────────────────────────────────┤')
print(f'│ 1. Accuracy:          {accuracy:.4f}                                        │')
print(f'│ 2. Precision:         {precision:.4f}                                        │')
print(f'│ 3. Recall:            {recall:.4f}                                        │')
print(f'│ 4. F1 Score:          {f1:.4f}                                        │')
print(f'│ 5. Log Loss:          {logloss:.4f}                                        │')
print('├─────────────────────────────────────────────────────────────────────┤')
print('│ ROC/AUC METRICS                                                     │')
print('├─────────────────────────────────────────────────────────────────────┤')
print(f'│ 6. AUC-ROC:           {auc_score:.4f}                                        │')
print(f'│    TPR:               {tpr[optimal_idx]:.4f}                                        │')
print(f'│    TNR:               {tnr[optimal_idx]:.4f}                                        │')
print(f'│    FPR:               {fpr[optimal_idx]:.4f}                                        │')
print(f'│    FNR:               {fnr[optimal_idx]:.4f}                                        │')
print('├─────────────────────────────────────────────────────────────────────┤')
print('│ ERROR METRICS                                                       │')
print('├─────────────────────────────────────────────────────────────────────┤')
print(f'│ 8a. MAE:              {mae:.6f}                                      │')
print(f'│ 8b. MSE:              {mse:.6f}                                      │')
print(f'│ 8c. RMSE:             {rmse:.6f}                                      │')
print(f'│ 8d. RMSLE:            {rmsle:.6f}                                      │')
print(f'│ 8e. R²:               {r2:.6f}                                      │')
print('├─────────────────────────────────────────────────────────────────────┤')
print('│ CLUSTERING METRICS                                                  │')
print('├─────────────────────────────────────────────────────────────────────┤')
try:
    print(f'│ 9a. Silhouette:       {silhouette:.4f}                                        │')
    print(f'│ 9b. Davies-Bouldin:   {dbi:.4f}                                        │')
except:
    print('│ 9a. Silhouette:       N/A                                           │')
    print('│ 9b. Davies-Bouldin:   N/A                                           │')
print('└─────────────────────────────────────────────────────────────────────┘')

In [ ]:
# =============================================================================
# Detailed Classification Report
# =============================================================================

print('\n' + '=' * 70)
print('DETAILED CLASSIFICATION REPORT')
print('=' * 70)

target_names = ['Normal (0)', 'Abnormal (1)']
print(classification_report(y_true_classes, y_pred_classes, target_names=target_names))

---

## Step 9: Prediction Function & Testing

**What this does**: Provides a function to make predictions on new ECG signals.

**For non-technical readers**: This is like the "use" button - you can input a new ECG reading and get a diagnosis prediction.

In [ ]:
# =============================================================================
# STEP 9: Prediction Function
# =============================================================================

def predict_ecg(signal, model, scaler, label_encoder):
    """
    Predict the class of an ECG signal.
    
    Parameters:
    - signal: 1D array of 188 ECG values
    - model: trained Keras model
    - scaler: fitted StandardScaler
    - label_encoder: fitted LabelEncoder
    
    Returns:
    - predicted_label: 0 (normal) or 1 (abnormal)
    - confidence: probability of the predicted class
    """
    # Reshape and normalize
    signal = np.array(signal).reshape(1, -1)
    signal_scaled = scaler.transform(signal)
    signal_scaled = signal_scaled.reshape(1, -1, 1)
    
    # Predict
    y_pred = model.predict(signal_scaled, verbose=0)
    predicted_class = np.argmax(y_pred)
    confidence = np.max(y_pred)
    predicted_label = label_encoder.inverse_transform([predicted_class])[0]
    
    return predicted_label, confidence

In [ ]:
# Test prediction with sample data
sample_data = [
    952, 954, 956, 955, 955, 953, 952, 952, 951, 955, 953, 954, 952, 953, 952, 955,
    957, 958, 958, 962, 963, 964, 963, 965, 963, 967, 969, 971, 973, 973, 972, 971,
    973, 973, 972, 968, 966, 968, 970, 973, 969, 966, 960, 964, 965, 970, 971, 970,
    967, 964, 962, 961, 960, 954, 950, 951, 951, 951, 950, 950, 950, 948, 952, 949,
    949, 944, 940, 943, 944, 947, 948, 944, 941, 943, 945, 945, 945, 942, 938, 936,
    933, 926, 927, 920, 910, 909, 919, 940, 963, 986, 1020, 1068, 1121, 1167, 1193,
    1201, 1182, 1136, 1069, 1010, 967, 939, 924, 914, 917, 924, 931, 934, 934, 937,
    940, 942, 942, 941, 940, 939, 940, 938, 938, 936, 934, 934, 938, 938, 938, 935,
    935, 936, 939, 938, 938, 939, 935, 935, 938, 937, 937, 935, 935, 937, 938, 939,
    938, 938, 936, 940, 938, 939, 938, 936, 936, 938, 938, 941, 939, 938, 933, 935,
    935, 938, 938, 936, 936, 937, 938, 941, 941, 939, 939, 940, 943, 943, 941, 941,
    939, 938, 943, 943, 943, 943, 938, 941, 942, 941, 938, 935, 931, 932
]

predicted_label, confidence = predict_ecg(sample_data, model, scaler, le)
print(f'Prediction: {"Normal" if predicted_label == 0 else "Abnormal"} (class {predicted_label})')
print(f'Confidence: {confidence:.4f} ({confidence*100:.2f}%)')

---

## Step 10: Save Model

**What this does**: Saves the trained model for future use.

In [ ]:
# =============================================================================
# STEP 10: Save Model
# =============================================================================

model.save('ecg_cnn_v2_final.keras')
print('Model saved successfully as ecg_cnn_v2_final.keras')

# Also save the scaler for future predictions
import joblib
joblib.dump(scaler, 'scaler_v2.pkl')
print('Scaler saved successfully as scaler_v2.pkl')

---

## Summary

This notebook implemented a **CNN-based ECG classifier** with:

✅ **Proper data normalization** using StandardScaler
✅ **80-10-10 train-validation-test split** with stratification
✅ **Comprehensive evaluation metrics** including:
   - Classification: Accuracy, Precision, Recall, F1, Log Loss
   - ROC Analysis: AUC, TPR, TNR, FPR, FNR with visualizations
   - Error Metrics: MAE, MSE, RMSE, RMSLE, R²
   - Clustering Quality: Silhouette Score, Davies-Bouldin Index

**Model Architecture**: 4-block CNN with batch normalization and dropout for regularization.